# 05 — PRACTICE (start here) — a gentle dry run of the exam

**Read this first. No rush.** This notebook is a mini practice exam. It has two questions — one of each type you'll see on Saturday — and it walks you through answering them using the kit, one small step at a time.

## The only routine you need to remember
Every real exam question is the same five moves:

1. **Read** the question.
2. **Find** the matching kit notebook (the playbook, notebook 00, has a lookup table).
3. **Copy** the cell(s) from that kit notebook into your answer.
4. **Change one line** — the line that points at the data file — and **run** it (Shift+Enter).
5. **Read** the output and **write** a short explanation using the "What to report" notes under that cell.

That's it. You are not writing code from scratch. You are picking the right pre-built cell, pointing it at the data, running it, and explaining what came out.

## Two things to know about where this runs
- **Questions of the "critique" type (notebook 01)** run in plain Python — you can practice them anywhere, even without Spark.
- **Questions of the "write code" type (notebooks 02, 03, 04)** need Spark, so practice those **on Jojie** (clone your repo there, same as the real exam).

---
# PRACTICE QUESTION 1 — the "critique" type (like Problem 2)

> An AI was asked to write code that samples **1% of devices**. Here is its answer. Give **3 things right** and **3 things wrong**, with justification.
>
> ```python
> TARGET_PERCENTAGE = 1
> MODULUS_BASE = 100
> def should_sample(tracker_id):
>     bucket = hash(tracker_id) % TARGET_PERCENTAGE   # % 1
>     return bucket < MODULUS_BASE                     # < 100
> ```

### How to answer it (follow along)
- **Step 1 — which kit page?** This is a *hash sampling* critique → **notebook 01**, the "Hash-based sampling" section.
- **Step 2 — run the verifier** to see the bug with your own eyes. The cell below is copied straight from notebook 01. Just run it (Shift+Enter).

In [ ]:
# (copied from notebook 01) — prove what the buggy code actually does
import hashlib
def h_int(x, seed=42):
    return int(hashlib.md5(f'{seed}:{x}'.encode()).hexdigest(), 16)

ids = [f"tracker_{i}" for i in range(100_000)]

def buggy(t):    return (h_int(t) % 1) < 100      # the AI's logic: %1 is always 0, 0<100 always True
def correct(t):  return (h_int(t) % 100) < 1      # the fix: 1-in-100

print("AI's code keeps:", f"{sum(buggy(t)   for t in ids)/len(ids):.0%}",  "<-- claims 1%!")
print("Correct code keeps:", f"{sum(correct(t) for t in ids)/len(ids):.2%}", "(target 1%)")

- **Step 3 — write your answer.** Double-click the cell below and fill in the blanks. There's a model answer right after, but *try it yourself first.*

**YOUR ANSWER (double-click to edit):**

*Three things right:*
1. It uses a hash of the ID, so the decision is reproducible without storing a list — ...
2. It runs in O(1) time and O(1) memory — ...
3. It does give explicit parameter values — ...

*Three things wrong:*
1. `hash % TARGET_PERCENTAGE` with `TARGET_PERCENTAGE=1` is always 0, so ...
2. The condition `bucket < 100` is then always true, so it samples ___% not 1% (I ran it: ___%).
3. The two constants are swapped — the correct logic is `hash % 100 < 1`. Also the seed isn't pinned, so ...

<details>
<summary><b>Model answer — open only after you've tried</b></summary>

**Right:** (1) deterministic hashing gives reproducible sampling with no stored list; (2) O(1) time and memory per event; (3) parameters are stated explicitly.
**Wrong:** (1) `hash % 1` is always 0, so `bucket` is always 0; (2) `0 < 100` is always true → it samples **100%**, not 1% (verified above); (3) the modulus and threshold are swapped — correct is `hash % 100 < 1`; the hash seed should also be pinned for cross-machine reproducibility.

</details>

**That's the whole skill for 40% of the exam:** read their answer → run the matching verifier in notebook 01 → write 3 right / 3 wrong citing the number you saw.

---
# PRACTICE QUESTION 2 — the "write code" type (like Problem 3a)

> Using Apache Spark, compute the **PageRank** of all nodes in the clickstream and show the **top 25**.

### How to answer it (follow along)
- **Step 1 — which kit page?** This is *PageRank* → **notebook 03**, the PageRank cell.
- **Step 2 — copy the cells, change one line.** In the real exam you'd copy notebook 03's Spark-setup cell, its load cell, and its PageRank cell into your answer, and set `DATA_PATH` to the exam's file. Here it's already pointed at the practice data. **Run the cell below** (needs Spark — do this on Jojie).

In [ ]:
# (copied from notebook 03) — Spark PageRank on the practice clickstream
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder.appName("practice-pagerank")
         .master("local[*]").config("spark.ui.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

DATA_PATH = "data/pageviews_practice.csv"      # <-- in the exam, change this ONE line to their file

raw = spark.read.csv(DATA_PATH, sep="\t", header=False, inferSchema=True).toDF("src","dst","type","count")
links = raw.filter(F.col("type")=="link").select("src","dst").dropDuplicates().cache()
nodes = links.select(F.col("src").alias("node")).union(links.select(F.col("dst").alias("node"))).distinct().cache()
N = nodes.count()

d = 0.85
outdeg = links.groupBy("src").agg(F.count("*").alias("outdeg")).cache()
ranks = nodes.withColumn("rank", F.lit(1.0/N))
for it in range(15):
    contribs = (links.join(ranks, links.src==ranks.node).join(outdeg,"src")
                .select(F.col("dst").alias("node"), (F.col("rank")/F.col("outdeg")).alias("c")))
    incoming = contribs.groupBy("node").agg(F.sum("c").alias("inc"))
    dangling = (ranks.join(outdeg, ranks.node==outdeg.src, "left_anti").agg(F.sum("rank")).first()[0]) or 0.0
    ranks = (nodes.join(incoming,"node","left").fillna(0.0,subset=["inc"])
             .withColumn("rank",(1-d)/N + d*(F.col("inc")+dangling/N)).select("node","rank"))

print("Top 25 by PageRank:")
ranks.orderBy(F.desc("rank")).limit(25).show(25, truncate=False)
spark.stop()

- **Step 3 — read the output and write it up.** You should see **United_States, India, World_War_II…** at the top. Then write two or three sentences, e.g.:

> "Computed PageRank with damping d=0.85 over 15 iterations, redistributing dangling-node mass so ranks sum to 1. The top nodes are United_States, India, and World_War_II — broad hub topics that many pages link to, consistent with a heavy-tailed in-degree."

**That's the whole skill for the code half:** find the cell → change the file path → run → read the top rows → describe them in plain sentences.

---
# You've now done one of each type. That's the whole exam.

Everything else is the same two motions with a different cell:
- **Critique questions** (Bloom, reservoir, HyperLogLog, running IQR, LSH) → notebook 01 / 02, run the verifier, write 3 right / 3 wrong.
- **Code questions** (HITS, Girvan-Newman, SimRank, clustering coefficient, find_similar) → notebooks 03 / 04, copy the cell, change the file path, run, describe the output.

## Practice plan before Saturday (about 90 minutes)
1. **(10 min)** Read notebook 00 (the playbook) once — just skim the lookup table so you know which notebook holds what.
2. **(20 min)** Do the two questions above yourself, from scratch, without peeking at the model answers.
3. **(30 min, on Jojie)** Open notebook 03 and 04, run each cell top to bottom on the practice data. Watch them work. If one errors, that's good — you learned it here, not Saturday.
4. **(20 min)** Read three critique sections in notebook 01 (Bloom, HyperLogLog, reservoir) and run their verifiers.
5. **(10 min)** Practice the exam-start routine once: on Jojie, `git clone` your repo, open notebook 00.

If any cell confuses you, that's a great thing to bring back to me before the exam. You do not need to understand every line — you need to know which cell to grab and which line to change. That's what this kit is for.